# Telco Customer Churn — Modeling

## Objective
Train and evaluate a baseline model and an improved model for churn prediction.

**Business priority:** maximize recall on churners (class `1`) to avoid missing at-risk customers, while monitoring precision to control retention costs.

**Metrics reported**
- ROC-AUC (ranking quality)
- Recall / precision for churn class
- Confusion matrix (business trade-off: FN vs FP)


In [ ]:
import sys
sys.path.append("..")

from src.data_prep import load_telco, get_feature_lists, make_split, build_preprocessor

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


In [ ]:
# Load the Telco Customer Churn dataset
# - Downloads the dataset from Kaggle if not already present locally
# - Creates a stable binary target variable (churn_flag)
# - Applies minimal cleaning (e.g. TotalCharges conversion)
df = load_telco()

# Retrieve the list of selected features based on business insights
# - cat_features: categorical variables to be one-hot encoded
# - num_features: numerical variables to be scaled
cat_features, num_features = get_feature_lists()

# Split the dataset into training and test sets
# - Stratified split to preserve churn rate distribution
# - Split is performed BEFORE preprocessing to avoid data leakage
X_train, X_test, y_train, y_test = make_split(df)

# Build the preprocessing pipeline
# - Numerical features: median imputation + standard scaling
# - Categorical features: mode imputation + one-hot encoding
# The resulting preprocessor will be reused consistently
# across modeling and the Streamlit application
preprocessor = build_preprocessor(cat_features, num_features)



## 1. Baseline Model — Logistic Regression
A strong, interpretable baseline to establish a reference performance level.
We use `class_weight="balanced"` to mitigate class imbalance and improve churn recall.


In [ ]:
# ------------------------------------------------------------
# Baseline model: Logistic Regression
# Purpose:
# - Provide a strong and interpretable baseline
# - Establish a reference performance level for churn prediction
# ------------------------------------------------------------

# Build a full modeling pipeline combining:
# - the preprocessing steps (imputation, scaling, encoding)
# - the classification model
# Using a Pipeline ensures consistency and prevents data leakage
baseline_clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"  # Handle class imbalance and improve churn recall
    ))
])

# Train the baseline model on the training set
baseline_clf.fit(X_train, y_train)

# Generate predictions on the test set
# - y_pred  : predicted class labels
# - y_proba : predicted probabilities for the churn class (class 1)
y_pred = baseline_clf.predict(X_test)
y_proba = baseline_clf.predict_proba(X_test)[:, 1]

# Evaluate model performance using churn-oriented metrics
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 4))
print("\nClassification report:\n", classification_report(y_test, y_pred))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))


## 2. Improved Model — Random Forest
A non-linear model that can capture interactions between contract type, tenure, services, and pricing.
Goal: improve churn recall and overall discrimination (ROC-AUC) compared to the baseline.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )),
])

rf_clf.fit(X_train, y_train)

y_pred_rf = rf_clf.predict(X_test)
y_proba_rf = rf_clf.predict_proba(X_test)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_proba_rf), 4))
print("\nClassification report:\n", classification_report(y_test, y_pred_rf))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred_rf))


## 3. Conclusion (Model Selection)
We select the model that best aligns with the churn prevention objective (high churn recall),
while keeping precision at an acceptable level for operational outreach.

Next steps:
- Threshold tuning to optimize the recall/precision trade-off
- Export model artifacts (`.joblib`) for the Streamlit app
- Add feature importance (business interpretability)
